# # Install dependencies

In [1]:
!pip install -q --user scikit-learn



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
ERROR: Can not perform a '--user' install. User site-packages are not visible in this virtualenv.


# # Load training data

In [2]:
"""Load the training data.

Pick the form that matches your storage:

    df = data.connection(conn_name).load("SELECT * FROM iris")   # SQL Explorer connection
    df = data.load("/home/jovyan/data/iris.csv")                 # local file (csv/parquet)
    df = data.load("s3://my-bucket/iris.parquet")                # S3 object

`data.split` shuffles rows by default (use `shuffle=False` to keep order,
e.g. for time-series). `random_state` makes the split reproducible.

`prepare()` is demo scaffolding — replace with your own data source in production.
"""
from src.demo import prepare
conn_name = prepare()

from nubison_model import data

df = data.connection(conn_name).load("SELECT * FROM iris")
datasets = data.split(df, {"train": 0.6, "val": 0.2, "test": 0.2}, random_state=42)

for name, sub in datasets.items():
    print(f"{name:6s} rows={len(sub):3d}")


/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: 

Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.


  color_warning(


train  rows= 90

val    rows= 30

test   rows= 30

# # Train (sklearn / xgboost / lightgbm / keras / skorch)

In [3]:
# `train(datasets, target, *, ...)` parameters:
#   datasets      — dict from `data.split` (must include "train";
#                   "val" enables `t.X_val / t.y_val` + auto-scoring;
#                   "test" populates `t.X_test / t.y_test`)
#   target        — label column name(s); single string or list of strings.
#                   Splits each split's DataFrame into X = features and
#                   y = target so estimators can call `fit(X, y)`.
#   model_type    — free-form string tagged on the run as `model_type`
#                   (surfaced in the nubison UI). "classifier" and
#                   "regressor" additionally make `t.fit()` log
#                   `val_accuracy` / `val_r2`. None skips the tag.
#   weights_path  — where `t.save(model)` writes the pickle.
#                   Default "src/weights.pkl" matches `register(artifact_dirs="src")`.
from nubison_model import train
from sklearn.linear_model import LogisticRegression

with train(datasets=datasets, target=["target"], model_type="classifier") as t:
    t.fit(LogisticRegression(max_iter=100))

print(f"run_id: {t.run_id}")


2026/05/18 11:32:09 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


2026/05/18 11:32:10 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/05/18 11:32:12 WARNING mlflow.sklearn.utils: roc_auc_score failed. The metric training_roc_auc will not be recorded. Metric error: too many indices for array: array is 2-dimensional, but 3 were indexed


/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run secretive-doe-416 at: http://127.0.0.1:5050/#/experiments/0/runs/b935bff01144418d9b889c69ca0b0b47


🧪 View experiment at: http://127.0.0.1:5050/#/experiments/0


run_id: b935bff01144418d9b889c69ca0b0b47